# Надёжный full-batch grokking на $S_5$ и $S_6$

Это отдельный протокол, точно повторяющий опубликованную реализацию: bias-free MLP, `torch.randperm` split 40/60, full-batch AdamW, FP64 cross-entropy. Запускайте сначала только $S_5$.

## 1. Загрузка актуального скрипта

Добавьте папку как Kaggle Dataset/Input. Ячейка печатает build ID и путь; если build не новый, запуск останавливается.

In [ ]:
from pathlib import Path
import shutil, sys

EXPECTED_BUILD = "sn-fullbatch-paper-v1.1-2026-08-02"
candidates = list(Path("/kaggle/input").rglob("train_sn_fullbatch.py"))
if not candidates:
    raise FileNotFoundError("Добавьте новый train_sn_fullbatch.py как Kaggle Input")
source = candidates[0]
local = Path("/kaggle/working/train_sn_fullbatch.py")
shutil.copy2(source, local)
sys.path.insert(0, "/kaggle/working")
from train_sn_fullbatch import BUILD_ID, Config, run
print("Script:", source)
print("Build:", BUILD_ID)
assert BUILD_ID == EXPECTED_BUILD, (BUILD_ID, EXPECTED_BUILD)

## 2. Конфигурация $S_5$

Старые minibatch checkpoints несовместимы и не импортируются благодаря отдельному имени протокола.

In [ ]:
CONFIG = Config(
    output_root="/kaggle/working/sn_fullbatch_grokking",
    protocol_name="stander_exact_fullbatch_v1",
    n_values=(5,),
    seeds=(42,),
    train_fraction=0.40,
    max_steps_by_n={5: 250_000, 6: 50_000},
    learning_rate=1e-3,
    weight_decay=1.0,
    betas=(0.9, 0.98),
    log_every=20,
    diagnostic_every=1_000,
    checkpoint_every=1_000,
    required_gap_steps=10_000,
    use_amp=False,
    resume_search_roots=("/kaggle/input",),
)
CONFIG

In [ ]:
RUN_DIRS = run(CONFIG)
RUN_DIRS

## 3. Кривые и последние строки логов

In [ ]:
import json, pandas as pd
import matplotlib.pyplot as plt

for run_dir in RUN_DIRS:
    frame = pd.read_csv(run_dir / "training_log.csv")
    display(frame.tail())
    result = json.loads((run_dir / "COMPLETED.json").read_text())
    print(json.dumps(result, indent=2))
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(frame.step, frame.train_loss, label="train loss")
    axes[0].plot(frame.step, frame.val_loss, label="val loss")
    axes[0].set_yscale("log"); axes[0].legend(); axes[0].grid(alpha=.25)
    axes[1].plot(frame.step, frame.train_acc, label="train acc")
    axes[1].plot(frame.step, frame.val_acc, label="val acc")
    axes[1].legend(); axes[1].grid(alpha=.25)
    plt.show()

## 4. Затем $S_6$

Запускайте в отдельной Kaggle-сессии: `CONFIG.n_values=(6,)`. Для продолжения добавьте output предыдущей Saved Version как Input. Скрипт сам найдёт совместимый checkpoint.

In [ ]:
# CONFIG.n_values = (6,)
# RUN_DIRS = run(CONFIG)